In [1]:
## Imports ##

# System Path #
import os
import sys 

# Add dsci_550_a1 to base path. Lets you project functions #
parent_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(parent_dir)

# Misc Data Handling #
import pandas as pd
import time
import re
import json 
from datetime import date 



# Runtime #
import time
from tqdm import tqdm 

# Iterators #
import collections
from itertools import chain
import ast
import random

# Flight Trajectory Functions #
from dsci_550_a1.flightFunctions import *

# Plotly #
import plotly.graph_objects as go
import plotly.express as px
import textwrap


In [ ]:
## Data ##

# Open Flights
df_american_routes = pd.read_csv("../data/joined_datasets/american_routes.tsv", sep = "\t")
df_american_routes["Flight_Path"] = df_american_routes["Flight_Path"].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)

# Our Airports
df_american_airports = pd.read_csv("../data/joined_datasets/american_airports.tsv", sep = "\t")
df_american_airports["Airport_Radius"] = df_american_airports["Airport_Radius"].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)

# Haunted Places Dataset 
df_haunted_places = pd.read_csv("../data/processed/haunted_places_features_added_v2.tab", sep = "\t")

# Flight Path Intersections
with open("../data/processed/flight_proximity_data.json", "r") as f:
    flight_intersection_data = json.load(f) 
# Airport Intersections
with open("../data/processed/airport_proximity_data.json", "r") as f:
    airport_intersection_data = json.load(f)


## Filter df_haunted_places by user specified "flagged_values" in "filter_colums"
user_input = {
    "Event_Type": ["Plane_Crash", "Flying_Object", "Electronic_Malfunction"]
}

filter_col = 'Event_Type'
flagged_values = ['Plane_Crash', 'Flying_Object', 'Electronic_Malfunction']

for key, values in user_input.items():
    df_haunted_places_filtered = df_haunted_places[df_haunted_places[key].apply(lambda x: any(val in x for val in values))].drop_duplicates()

    # If column is a list with multiple possible values, we assign a single value in the order of "values"
    
    if df_haunted_places_filtered[key].dtype == object and df_haunted_places_filtered[key].str.contains(r'\|', na=False).any():
        
        for idx in df_haunted_places_filtered.index.tolist():
            x = df_haunted_places_filtered.loc[idx, f'{key}']
            
            while x not in values:
                
                for val in values:
                    
                    if val in x:
                        df_haunted_places_filtered.loc[idx, f'{key}'] = val
                     s   x = val
                        break
    



In [ ]:
# Open Flights
df_american_routes = pd.read_csv("../data/joined_datasets/american_routes.tsv", sep = "\t")
df_american_routes["Flight_Path"] = df_american_routes["Flight_Path"].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)

# Our Airports
df_american_airports = pd.read_csv("../data/joined_datasets/american_airports.tsv", sep = "\t")
df_american_airports["Airport_Radius"] = df_american_airports["Airport_Radius"].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)

# Haunted Places Dataset 
df_hp = pd.read_csv("../data/processed/haunted_places_features_added_v2.tab", sep = "\t")
df_hp['Haunted_Places_Date'] = df_hp['Haunted_Places_Date'].apply(lambda x: ast.literal_eval(x)) 
df_hp['Haunted_Places_Date'] = df_hp['Haunted_Places_Date'].apply(lambda x: [parse_date(y) for y in x] if isinstance(x, list) else x)


# Flight Path Intersections
with open("../data/processed/flight_proximity_data.json", "r") as f:
    flight_intersection_data = json.load(f) 
# Airport Intersections
with open("../data/processed/airport_proximity_data.json", "r") as f:
    airport_intersection_data = json.load(f)

In [ ]:
## Query ##

# Date Range Filter
def parse_date(s): 
    return date(*map(int, s.split('-'))) 

def in_date_range(date_list, start_date, end_date):
    return any(start_date <= date <= end_date for date in date_list)

# Main Query Function
def filter_haunted_data(state=None, event_type=None, apparition_type=None, haunt_date_range=None):
    df_filtered = df_hp.copy()
    
    if state:
        df_filtered = df_filtered[df_filtered['State'] == state]
    if event_type:
        df_filtered = df_filtered[df_filtered['Event_Type'].str.contains(event_type, na=False)]
    if apparition_type:
        df_filtered = df_filtered[df_filtered['Apparition_Type'].str.contains(apparition_type, na=False)]
    if haunt_date_range:
        start_date, end_date = map(parse_date, haunt_date_range)
        df_filtered = df_filtered[(df_filtered['Haunted_Places_Date'].apply(lambda x: in_date_range(x, start_date, end_date)))]

    haunted_ids = df_filtered['Haunted_Places_Id'].astype(str).tolist()

    filtered_flight_intersections = {k: v['Routes'] for k, v in flight_intersection_data.items() if k in haunted_ids}
    filtered_airport_intersections = {k: v['Airports'] for k, v in airport_intersection_data.items() if k in haunted_ids}

    relevant_routes = set()
    relevant_iata_codes = set()
    relevant_airports = set()

    for _, v in filtered_flight_intersections.items():
        relevant_routes.update(route['Route_ID'] for route in v)
        relevant_iata_codes.update(
            chain(
            (route['Dest_Airport'] for route in v),
            (route['Source_Airport'] for route in v)
            )
        )
        
    for _, v in filtered_airport_intersections.items():
        relevant_airports.update(airport['Airport_ID'] for airport in v)

    routes_filtered = df_american_routes.loc[list(relevant_routes)]
    airports_filtered = df_american_airports[df_american_airports['Id'].isin(relevant_airports) | df_american_airports['Iata_Code'].isin(relevant_iata_codes)]

    return df_filtered, routes_filtered, airports_filtered

df_filtered, df_routes, df_airports = filter_haunted_data(
    haunt_date_range=("1000-10-31", "1000-10-31")
)

print("Filtered Haunted Places:", len(df_filtered))
print("Associated Routes:", len(df_routes))
print("Nearby Airports:", len(df_airports))


In [2]:
# Helper Functions #
from dsci_550_a3.dg_viz import hp_interactive_globe
from dsci_550_a3.dg_query import filter_hp_df, get_legend_items
from dsci_550_a3.dg_dataLoader import load_all_data


## Load Data and Define Holidays
(
    hp_df,
    route_df,
    airport_df,
    flight_intersections,
    airport_intersections
) = load_all_data()




In [5]:


from itertools import chain
from datetime import date

# Unique Legend Items
def get_legend_items(df_hp, legend_key):
    s = set().union(*df_hp[legend_key].dropna().str.split(' | ').tolist())
    try:
        s.remove('|')  # remove delimiter if it was caught
    except:
        pass
    return  list(s)

# Date Range Filter
def parse_date(s): 
    return date(*map(int, s.split('-'))) 

def in_date_range(date_list, start_date, end_date):
    return any(start_date <= date <= end_date for date in date_list)

# Main Query Function
def filter_hp_df(
    hp_df,
    route_df,
    airport_df,
    flight_intersections,
    airport_intersections,
    state=None, event_type=None, apparition_type=None, haunt_date_range=None, holiday = None):
    
    filtered_hp_df = hp_df.copy()
    
    if state:
        filtered_hp_df = filtered_hp_df[filtered_hp_df['State'] == state]
    if event_type:
        filtered_hp_df = filtered_hp_df[filtered_hp_df['Event_Type'].str.contains(event_type, na=False)]
    if apparition_type:
        filtered_hp_df = filtered_hp_df[filtered_hp_df['Apparition_Type'].str.contains(apparition_type, na=False)]
    if haunt_date_range:
        start_date, end_date = map(parse_date, haunt_date_range)
        filtered_hp_df = filtered_hp_df[(filtered_hp_df['Haunted_Places_Date'].apply(lambda x: in_date_range(x, start_date, end_date)))]
    if holiday:
        holiday = parse_date(holiday)
        filtered_hp_df = filtered_hp_df[filtered_hp_df['Haunted_Places_Date'].apply(lambda x: in_date_range(x, holiday, holiday))]

    haunted_ids = filtered_hp_df['Haunted_Places_Id'].astype(str).tolist()

    filtered_flight_intersections = {k: v['Routes'] for k, v in flight_intersections.items() if k in haunted_ids}
    filtered_airport_intersections = {k: v['Airports'] for k, v in airport_intersections.items() if k in haunted_ids}

    relevant_routes = set()
    relevant_iata_codes = set()
    relevant_airports = set()

    for _, v in filtered_flight_intersections.items():
        relevant_routes.update(route['Route_ID'] for route in v)
        relevant_iata_codes.update(
            chain(
            (route['Dest_Airport'] for route in v),
            (route['Source_Airport'] for route in v)
            )
        )
        
    for _, v in filtered_airport_intersections.items():
        relevant_airports.update(airport['Airport_ID'] for airport in v)

    filtered_route_df = route_df.loc[list(relevant_routes)]
    filtered_airport_df = airport_df[airport_df['Id'].isin(relevant_airports) | airport_df['Iata_Code'].isin(relevant_iata_codes)]

    return filtered_hp_df, filtered_route_df, filtered_airport_df




legend_arg = None
state = None
event_type = None
apparition_type = None
haunt_date_range = None
holiday = '1000-10-31'


if not legend_arg:
    legend_arg = {'Event_Type': get_legend_items(hp_df, 'Event_Type')} 

filtered_hp_df, filtered_route_df, filtered_airport_df = filter_hp_df(
    hp_df,
    route_df,
    airport_df,
    flight_intersections,
    airport_intersections,
    state,
    event_type,
    apparition_type,
    haunt_date_range,
    holiday,
)
# holiday = parse_date(holiday)
# in_date_range(hp_df['Haunted_Places_Date'][0], holiday, holiday)
filtered_hp_df

,Haunted_Places_Id,City,Country,Description,Location,State,State_Abbrev,Longitude,Latitude,City_Longitude,...,Distance_to_Nearest_Worship,Religion_Intersection,Daylight_Duration_Hours,Named_Entities,Image_Pointer,Image_Caption,Image_Objects,GeoTopic_Locations,GeoTopic_Latitudes,GeoTopic_Longitudes
9,9,Allegan,United States,Various ghostly activities. News coverage abou...,The Grill House and the Rock Bottom Bar,Michigan,MI,-85.857564,42.497762,-85.855303,...,4678.88,NaN,9.186,"[('every year', 'DATE'), ('Halloween', 'DATE')]",hpimg_9.png,a black and white photo of a clock tower .,"rapeseed: 51.20%, church: 7.46%, barn: 7.30%, ...",Michigan,44.25029,-85.50033
361,361,Ossineke,United States,There is a story that this old lady that owned...,"Nickson Hill, Road Mansion",Michigan,MI,-83.442474,44.902234,-83.442474,...,688.44,christian,8.914,"[('night', 'TIME'), ('Halloween', 'DATE'), ('n...",hpimg_361.png,a black and white photo of a man and a dog,"totem_pole: 32.88%, book_jacket: 13.92%, comic...",Michigan,44.25029,-85.50033
428,428,Saginaw,United States,JB Meinburg's is an old pub that is said to be...,JB Meinburg's,Michigan,MI,-83.963815,43.416382,-83.950807,...,293.51,christian,9.085,"[(""JB Meinburg's"", 'ORG'), ('just about every ...",hpimg_428.png,a black and white photo of a woman sitting on ...,"book_jacket: 21.37%, barbershop: 6.31%, rockin...",NaN,NaN,NaN
450,450,St. Clair,United States,It is said that in the fall of the year - near...,Pug Road,Michigan,MI,-82.487393,42.854684,-82.486024,...,3650.41,NaN,9.146,"[('the fall of the year', 'DATE'), ('Halloween...",hpimg_450.png,a black and white photo of a building,"barn: 85.96%, church: 7.14%, boathouse: 0.98%,...","Michigan, Saint Clair County","44.25029, 38.47031","-85.50033, -89.92841"
487,487,Vernon,United States,Geek Rd. - People have reported seeing a brig...,Corunna area,Michigan,MI,-84.029408,42.939197,-84.029408,...,4616.93,christian,9.138,"[('Geek Rd', 'FAC'), ('midnight', 'TIME'), ('H...",hpimg_487.png,a person is skiing down a snowy hill .,"umbrella: 15.73%, alp: 9.73%, ski: 9.63%, moun...",Michigan,44.25029,-85.50033
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10365,10365,Oxford,United States,Believed that in the 1900's sometime the caret...,Hookman's Cemetery,Connecticut,CT,-73.119009,41.383282,-73.116839,...,1769.85,christian,10.437,"[(""the 1900's"", 'DATE'), ('one', 'CARDINAL'), ...",NaN,NaN,No image available,Oxford County,44.49977,-70.75657
10397,10397,Unionville,United States,This cemetery is home to multiple hauntings. O...,Unionville Cemetery,Connecticut,CT,-72.891259,41.760746,-72.888846,...,6.65,christian,9.279,"[('One', 'CARDINAL'), ('mid-night', 'TIME'), (...",NaN,NaN,No image available,Connecticut,41.66704,-72.66648
10410,10410,West Hartford,United States,RM#22 - The ghost of a long lost child who was...,King Phillip Middle School,Connecticut,CT,-72.741271,41.794160,-72.742015,...,1090.84,jewish,9.265,"[('KPM', 'ORG'), ('22', 'CARDINAL'), ('every H...",NaN,NaN,No image available,"Connecticut, Kompiam Airport, United States","41.66704, -5.38146, 39.76","-72.66648, 143.92502, -98.5"
10427,10427,Woodbury,United States,The cemetery behind the church harbors an unwa...,Episcopal Church,Connecticut,CT,-73.208141,41.541017,-73.209002,...,26.54,christian,8.796,"[('Halloween', 'DATE'), ('night', 'TIME')]",NaN,NaN,No image available,"Connecticut, Saint Johns Episcopal Church","41.66704, 13.50776","-72.66648, 144.80922"


In [ ]:
get_legend_items(hp_df, 'Event_Type')